# Runtime Curve Walkthrough: Winter Storm x Solar

This notebook reads the proposed `winter_storm_solar` artifact and shows how a consumer such as the Hazard grid
would call it. The order is: build a payload, evaluate each failure unit, see the scenario bounds and the
module-structure cascade, check the rejections, then assemble a conditional loss on explicit values.

Start with `00_curve_curation_walkthrough.ipynb` to see where the numbers come from. This notebook covers damage
ratio and conditional loss only. Event frequency, EAL and PML belong to the Hazard tier.

In [1]:
from pathlib import Path
import json, math, sys
from math import erf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_colwidth", 120)

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "docs/cells").is_dir() and (p / "scripts/reference_helpers").is_dir():
            return p
    raise RuntimeError("run this notebook from inside the damage_modeling repository")

ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(ROOT / "scripts/reference_helpers"))
from pathway_damage_curve_eval import PathwayEvaluationError, evaluate_ordered_damage_state_record
from winter_storm_solar_curve_eval import evaluate, assemble_loss, ground_frozen_load_kpa

CELL = ROOT / "docs/cells/winter_storm_solar/proposed"
ART_PATH = CELL / "winter_storm_solar__model_v0_1__docs_r1__curve_artifact.json"
artifact = json.loads(ART_PATH.read_text())
capability = artifact["capability_declaration"]
pathway = artifact["pathways"][0]
records = {r["failure_unit_id"]: r for r in pathway["curve_records"]}
STRUCT = records["WSS1_SUPPORT_STRUCTURE_COLLAPSE"]
MODULE = records["WSS1_MODULE_FIELD_BREAKAGE"]
OUT = Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
SCEN = ["lower_resistance", "central_screening", "upper_resistance"]
STYLE = {"lower_resistance": ("#85B7EB", "--", "weak"), "central_screening": ("#2a78d6", "-", "middle"),
         "upper_resistance": ("#185FA5", ":", "strong")}
phi = lambda z: 0.5 * (1 + erf(z / math.sqrt(2)))
print(artifact["cell_id"], "|", artifact["semantic_damage_model_version"], "|", artifact["documentation_revision"],
      "|", artifact["promotion_status"], "| canonical:", artifact["canonical_runtime_artifact"])

winter_storm_solar | model v0.1 | docs r1 | proposed | canonical: False


## 1. The Artifact Contract

The required identity, axis contracts and the load bridge that turns SWE into kPa.

In [2]:
display(pd.DataFrame([{"field": k, "value": artifact[k]} for k in ("cell_id", "damage_code_id", "semantic_damage_model_version",
                       "documentation_revision", "lifecycle_state", "model_grade", "canonical_runtime_artifact")]))
display(pd.DataFrame(pathway["hazard_axis"]["axis_contracts"]).T)
display(pd.DataFrame([pathway["hazard_axis"]["load_bridge"]]).T.rename(columns={0: "value"}))

,field,value
0,cell_id,winter_storm_solar
1,damage_code_id,WINTER_STORM_SOLAR_SNOW_LOAD_V0_1_PROPOSED
2,semantic_damage_model_version,model v0.1
3,documentation_revision,docs r1
4,lifecycle_state,candidate
5,model_grade,screening_engineering_proxy
6,canonical_runtime_artifact,False


,axis_field,unit,valid_range,flag_above,definition
structure,event_to_design_snow_demand_ratio,dimensionless,"[0.0, 4.0]",3.3,event ground frozen load / design ground snow load (site_code_assumed) or event array load / rated array snow load (...
module,array_snow_load_kpa,kPa,"[0.0, 6.0]",4.5,Cs x 0.7 x Ce x Ct x event ground frozen load


,value
swe_to_kpa_per_mm,0.009807
glaze_kpa_per_mm_radial,0.027728
flat_roof_factor,0.7
exposure_factor_ce,0.9
thermal_factor_ct,1.2
default_slope_factor_cs,1.0
source_ids,"[WSS-S031, WSS-S002, WSS-S025]"
snow_depth_policy,reject snow depth without SWE (SNOW_DEPTH_WITHOUT_SWE_UNSUPPORTED)


## 2. One Evaluation Call

A screening-grid style payload: 150 mm SWE, no ice supplied by the hazard product, and racking assumed built to
a site-code design load of 0.96 kPa.

In [3]:
payload = {"pathway_id": "snow_gravity_load", "array_architecture": "single_axis_tracker",
           "snow_water_equivalent_peak_mm": 150, "ice_component_basis": "not_in_hazard_product",
           "racking_design_basis": "site_code_assumed", "design_ground_snow_load_kpa": 0.96}
for unit in ("WSS1_SUPPORT_STRUCTURE_COLLAPSE", "WSS1_MODULE_FIELD_BREAKAGE"):
    res = evaluate(artifact, {**payload, "failure_unit_id": unit})
    print(unit)
    print("  axis   :", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in res["axis"].items() if k != "flags"})
    print("  DRs    :", {k: round(v, 3) for k, v in res["scenario_drs"].items()})
    if "central_cascade_bounds" in res:
        print("  cascade:", {k: round(v, 3) for k, v in res["central_cascade_bounds"].items()}, "(central, salvage bounds)")
    print("  flags  :", res["flags"])

WSS1_SUPPORT_STRUCTURE_COLLAPSE
  axis   : {'ratio': 1.532, 'array_load_kpa': 1.112, 'ground_load_kpa': 1.471, 'payload_mode': 'swe_bridge_site_code_assumed'}
  DRs    : {'lower_resistance': 0.713, 'central_screening': 0.533, 'upper_resistance': 0.256}
  flags  : ['ICE_COMPONENT_NOT_SUPPLIED_LOWER_BOUND', 'DESIGN_BASIS_ASSUMED_SITE_CODE', 'TRACKER_TRANSFER_FROM_FIXED_RACK_TESTS']
WSS1_MODULE_FIELD_BREAKAGE
  axis   : {'ratio': 1.532, 'array_load_kpa': 1.112, 'ground_load_kpa': 1.471, 'payload_mode': 'swe_bridge_site_code_assumed'}
  DRs    : {'lower_resistance': 0.613, 'central_screening': 0.433, 'upper_resistance': 0.206}
  cascade: {'full_salvage': 0.012, 'no_salvage': 0.539} (central, salvage bounds)
  flags  : ['ICE_COMPONENT_NOT_SUPPLIED_LOWER_BOUND', 'DESIGN_BASIS_ASSUMED_SITE_CODE', 'TRACKER_TRANSFER_FROM_FIXED_RACK_TESTS', 'MODULE_SALVAGE_T4_ASSUMPTION']


## 3. Same Storm, Different Design Loads

The design load decides the outcome. This sweep is for the middle scenario.

In [4]:
rows = []
for design in (0.48, 0.96, 1.44, 2.39):
    p = {**payload, "design_ground_snow_load_kpa": design}
    s = evaluate(artifact, {**p, "failure_unit_id": "WSS1_SUPPORT_STRUCTURE_COLLAPSE"})
    m = evaluate(artifact, {**p, "failure_unit_id": "WSS1_MODULE_FIELD_BREAKAGE"})
    rows.append({"design_kPa": design, "r": round(s["axis"]["ratio"], 2), "racking DR": round(s["scalar_central_dr"], 3),
                 "module DR (with cascade)": round(m["scalar_central_dr"], 3)})
display(pd.DataFrame(rows))

,design_kPa,r,racking DR,module DR (with cascade)
0,0.48,3.06,0.937,0.753
1,0.96,1.53,0.533,0.433
2,1.44,1.02,0.222,0.188
3,2.39,0.62,0.034,0.039


## 4. Screening Grid vs Per-Asset Payload

It is the same curve with a different denominator. A per-asset run can supply the project's rated array snow
load. For example, a standard 5 psf tracker is rated about 0.24 kPa.

In [5]:
grid = {**payload, "snow_water_equivalent_peak_mm": 60, "failure_unit_id": "WSS1_SUPPORT_STRUCTURE_COLLAPSE"}
asset = {k: v for k, v in grid.items() if k != "design_ground_snow_load_kpa"}
asset.update({"racking_design_basis": "project_rated", "racking_rated_array_snow_load_kpa": 0.2394})
for name, p in (("screening (site code 0.96 kPa)", grid), ("per-asset (tracker rated 5 psf)", asset)):
    res = evaluate(artifact, p)
    print(f"{name:34s} r = {res['axis']['ratio']:.2f}  racking DR (middle) = {res['scalar_central_dr']:.3f}  flags = {res['flags']}")

screening (site code 0.96 kPa)     r = 0.61  racking DR (middle) = 0.034  flags = ['ICE_COMPONENT_NOT_SUPPLIED_LOWER_BOUND', 'DESIGN_BASIS_ASSUMED_SITE_CODE', 'TRACKER_TRANSFER_FROM_FIXED_RACK_TESTS']
per-asset (tracker rated 5 psf)    r = 1.86  racking DR (middle) = 0.686  flags = ['ICE_COMPONENT_NOT_SUPPLIED_LOWER_BOUND', 'TRACKER_TRANSFER_FROM_FIXED_RACK_TESTS']


## 5. Fail-Closed Behaviour

Invalid or unsupported calls are **rejected with a stable reason code**. They never get a guessed number.

In [6]:
cases = {
    "snow depth instead of SWE": {**{k: v for k, v in payload.items() if k != "snow_water_equivalent_peak_mm"}, "snow_depth_mm": 400},
    "no design basis": {k: v for k, v in payload.items() if k != "racking_design_basis"},
    "withheld unit (foundation)": {**payload, "failure_unit_id": "WSS1_FOUNDATION"},
    "deferred pathway (ice + wind)": {**payload, "pathway_id": "ice_with_concurrent_wind"},
    "roof-mounted array": {**payload, "array_architecture": "roof_mount"},
    "demand above valid range (150 mm on a 0.24 kPa design)": {**payload, "design_ground_snow_load_kpa": 0.24},
}
rows = []
for name, p in cases.items():
    p = {"failure_unit_id": "WSS1_SUPPORT_STRUCTURE_COLLAPSE", **p}
    try:
        evaluate(artifact, p); rows.append({"case": name, "result": "accepted (unexpected)"})
    except PathwayEvaluationError as e:
        rows.append({"case": name, "result": e.code})
display(pd.DataFrame(rows))

,case,result
0,snow depth instead of SWE,SNOW_DEPTH_WITHOUT_SWE_UNSUPPORTED
1,no design basis,RACKING_DESIGN_BASIS_REQUIRED
2,withheld unit (foundation),NO_RUNTIME_CURVE_FOR_PATHWAY_UNIT
3,deferred pathway (ice + wind),PATHWAY_DEFERRED_NOT_IMPLEMENTED
4,roof-mounted array,ARRAY_ARCHITECTURE_UNSUPPORTED
5,demand above valid range (150 mm on a 0.24 kPa design),AXIS_OUTSIDE_VALID_RANGE


## 6. Conditional Loss On An Explicit Value Basis

This uses the standard 120 MWdc tracker plant at the reference values, with the whole array exposed and the
support pool allocated once. The withheld units are not included, so this is **not** a full-plant loss.

In [7]:
values = dict(currency="USD", price_year=2024, value_unit="USD_per_kWdc", module_value=291.21485143992487,
              structure_value=109.98972602739727, capacity_kwdc=120_000, exposed_fraction=1.0, support_pool_value=189.59050092005714)
physical_total = 877.7957023626668 * 120_000
rows = []
for site, design in [("north Texas", 0.48), ("Missouri", 0.96), ("Minnesota", 2.39)]:
    base = {**payload, "ice_component_basis": "supplied", "ice_radial_thickness_mm": 0, "design_ground_snow_load_kpa": design}
    s = evaluate(artifact, {**base, "failure_unit_id": "WSS1_SUPPORT_STRUCTURE_COLLAPSE"})
    m = evaluate(artifact, {**base, "failure_unit_id": "WSS1_MODULE_FIELD_BREAKAGE"})
    for sid in SCEN:
        loss = assemble_loss(artifact, s["scenario_drs"][sid], m["scenario_drs"][sid], values)
        rows.append({"site (illustrative design)": f"{site} ({design} kPa)", "scenario": STYLE[sid][2],
                     "loss $M": round(loss["total_loss"] / 1e6, 1), "% of physical value": round(100 * loss["total_loss"] / physical_total, 1)})
table = pd.DataFrame(rows).pivot(index="site (illustrative design)", columns="scenario", values="% of physical value")[["weak", "middle", "strong"]]
display(table)
print("Heavy wet snow, 150 mm SWE; % of the plant's $877.80/kWdc physical value.")

scenario,weak,middle,strong
site (illustrative design),,,
Minnesota (2.39 kPa),8.5,2.2,0.3
Missouri (0.96 kPa),37.7,27.1,12.9
north Texas (0.48 kPa),50.2,47.3,39.6


Heavy wet snow, 150 mm SWE; % of the plant's $877.80/kWdc physical value.


## 7. What This Notebook Does Not Do

- No event frequency, EAL, PML, VaR or TVaR: those are Hazard-tier calculations.
- No foundation, electrical, SCADA or civil damage: those units are withheld.
- No business interruption: downtime from snow cover or cold belongs to performance and BI modelling.